# How do annual members and casual riders use Cyclistic bikes differently?
Project Title : **Cyclistic bike-share analysis**

Data Analyst: **Venkata Praveen Jasthi**

Company : **Cyclistic** (Chicago)

Client : **Cyclistic Marketing Team**

Sponsor : **Lily Moreno** (Director of Marketing)

**Purpose**: The director of marketing believes the company’s future success depends on maximizing the number of annual memberships. Therefore, your team wants to understand how casual riders and annual members use Cyclistic bikes differently. From these insights, your team will design a new marketing strategy to convert casual riders into annual members.


**Overview of the Notebook**

This notebooke includes includes all important steps for analysis

Prepare the data for exploration
Process the data from dirty to clean
Analyse the data to answer the questions

### Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import glob
import os
from google.colab import drive

### Connect to the file on drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
directory_path = "/content/drive/MyDrive/cyclistic_share_analysis/12_months_datasets/"
output_path    = "/content/drive/MyDrive/cyclistic_share_analysis/outputs/"
os.makedirs(output_path, exist_ok=True)

### Load and inspect the monthly data

In [ ]:
directory_path = "/content/drive/MyDrive/cyclistic_share_analysis/12_months_datasets/"
for file_name in os.listdir(directory_path):
  print(file_name)
all_files = glob.glob(os.path.join(directory_path, "*.csv"))
data = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
df = pd.DataFrame(data)

In [ ]:
print("Total files number :", len(all_files))

In [ ]:
print("Columns names :", df.columns)

In [ ]:
print(df.shape)

In [ ]:
print("Null values in all columns\n", df.isnull().sum())

In [ ]:
print(df.dtypes)

In [ ]:
print(df.head())

In [ ]:
print(df.info)

In [ ]:
(print(df.duplicated().sum()))

In [ ]:
print(df.isnull().sum())

### Prepare and clean the trip records

In [ ]:
print(df['rideable_type'].unique())
print(df['member_casual'].unique())

In [ ]:
print(df.dtypes)

In [ ]:
print(" sarted_at and end_at columns data type is change successfully  ")
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

In [ ]:
print(df.dtypes)

In [ ]:
df['ride_length'] = df['ended_at'] - df['started_at']
df['ride_length'] = df['ride_length'].dt.total_seconds() / 60
df['month'] = df['started_at'].dt.month_name()
df['day_of_week'] = df['started_at'].dt.day_name()
df['hour'] = df['started_at'].dt.hour

In [ ]:
print(df['month'].unique())
print(df['hour'].unique())
print(df['day_of_week'].unique())

In [ ]:
# Check invalid ride durations
print("Zero or negative rides:", (df['ride_length'] <= 0).sum())

# Remove invalid ride durations
df = df[df['ride_length'] > 0].copy()

print("Rows after removing invalid rides:", df.shape[0])

In [ ]:
# Rides under 1 minute are mostly false starts or people re-docking the bike.
# Anything over 24 hours is a lost/stolen bike or a system error, not a real trip.
too_short = (df["ride_length"] < 1).sum()
too_long = (df["ride_length"] > 1440).sum()
print("Rides under 1 minute:", too_short)
print("Rides over 24 hours:", too_long)

df = df[df["ride_length"].between(1, 1440)].copy()
print("Rows left after cleaning:", df.shape[0])

In [ ]:
# A few rides from 30 Sep 2025 are saved in the October file, so keep only the 12 months
before = len(df)
df = df[(df["started_at"] >= "2025-10-01") & (df["started_at"] < "2026-10-01")].copy()
print("Rides outside Oct 2025 - Sep 2026:", before - len(df))
print("Rows left after cleaning:", len(df))

In [ ]:
# Data quality check on the raw combined data, before cleaning
total = len(data)
print("Total rows:", total)

checks = {
    "Missing start station name": data["start_station_name"].isna().sum(),
    "Missing end station name": data["end_station_name"].isna().sum(),
    "Missing end lat/lng": data["end_lat"].isna().sum(),
    "Duplicate ride_id": data["ride_id"].duplicated().sum(),
}
for name, n in checks.items():
    print(f"{name}: {n:,} ({n / total:.1%})")

print(data["member_casual"].value_counts())

In [ ]:
# Count what each cleaning rule removes, starting again from the raw data
steps = []
d = data.copy()

n = len(d); d = d.dropna(subset=["end_lat", "end_lng"])
steps.append(("Missing end coordinates", n - len(d)))

d["started_at"] = pd.to_datetime(d["started_at"])
d["ended_at"] = pd.to_datetime(d["ended_at"])
d["ride_length"] = (d["ended_at"] - d["started_at"]).dt.total_seconds() / 60

n = len(d); d = d[d["ride_length"] > 0]
steps.append(("Zero or negative length", n - len(d)))

n = len(d); d = d[d["ride_length"] >= 1]
steps.append(("Shorter than 1 minute", n - len(d)))

n = len(d); d = d[d["ride_length"] <= 1440]
steps.append(("Longer than 24 hours", n - len(d)))

n = len(d); d = d[(d["started_at"] >= "2025-10-01") & (d["started_at"] < "2026-10-01")]
steps.append(("Started outside Oct 2025 - Sep 2026", n - len(d)))

print("Raw rows:", f"{len(data):,}")
for name, removed in steps:
    print(f"{name}: {removed:,}")
print("Rows left:", f"{len(d):,}")

In [ ]:
print("Rows after cleaning:", len(df))
print("Duplicate ride_id:", df["ride_id"].duplicated().sum())
print("Shortest / longest ride (min):", round(df["ride_length"].min(), 2), "/", round(df["ride_length"].max(), 2))
print("First / last ride:", df["started_at"].min(), "/", df["started_at"].max())
print("Rider types:", df["member_casual"].unique())
print("Bike types:", df["rideable_type"].unique())
print(df["member_casual"].value_counts(normalize=True).round(3))

In [ ]:
print(df.dtypes)

In [ ]:
print(df.isnull().sum())

# Analyse riding patterns

In [ ]:
# one colour per rider type, used in every chart from here on
colors = {"member": "#2E4A7D", "casual": "#E4572E"}

def save_chart(name):
    plt.savefig(output_path + name + ".png", dpi=150, bbox_inches="tight")

### Overall ride-duration statistics

In [ ]:
# The calculations the case study guide asks for: mean, max and mode
print("Mean ride length (min):", round(df["ride_length"].mean(), 2))
print("Max ride length (min):", round(df["ride_length"].max(), 2))
print("Most common day:", df["day_of_week"].mode()[0])

guide_stats = df.groupby("member_casual").agg(
    mean_minutes=("ride_length", "mean"),
    max_minutes=("ride_length", "max"),
    busiest_day=("day_of_week", lambda x: x.mode()[0]),
    busiest_hour=("hour", lambda x: x.mode()[0]),
)
print(guide_stats.round(2))

### Overall comparison of rider groups

In [ ]:
rider_summary = (
    df.groupby("member_casual")
      .agg(
          total_rides=("ride_id", "count"),
          average_minutes=("ride_length", "mean"),
          median_minutes=("ride_length", "median"),
          total_minutes=("ride_length", "sum")
      )
)

In [ ]:
rider_summary["ride_share_pct"] = (
    rider_summary["total_rides"]
    / rider_summary["total_rides"].sum() * 100
)

rider_summary["average_minutes"] = (
    rider_summary["average_minutes"].round(2)
)

rider_summary["median_minutes"] = (
    rider_summary["median_minutes"].round(2)
)

rider_summary["total_hours"] = (
    rider_summary["total_minutes"] / 60
).round(1)

In [ ]:
print(rider_summary.round(2))

### Differences by day of the week

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

daily_summary = (
    df.groupby(["day_of_week", "member_casual"])
      .agg(
          rides=("ride_id", "count"),
          average_minutes=("ride_length", "mean")
      )
      .reset_index()
)

In [ ]:
daily_summary["day_of_week"] = pd.Categorical(
    daily_summary["day_of_week"],
    categories=day_order,
    ordered=True
)

daily_summary = daily_summary.sort_values("day_of_week")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=daily_summary,
    x="day_of_week", y="rides",
    hue="member_casual", ax=axes[0]
)
axes[0].set_title("Number of rides by day")
axes[0].set_xlabel("")
axes[0].set_ylabel("Ride count")
axes[0].tick_params(axis="x", rotation=40)

sns.lineplot(
    data=daily_summary,
    x="day_of_week", y="average_minutes",
    hue="member_casual", marker="o", ax=axes[1]
)
axes[1].set_title("Average ride duration by day")
axes[1].set_xlabel("")
axes[1].set_ylabel("Minutes")
axes[1].tick_params(axis="x", rotation=40)

plt.tight_layout()
plt.show()

###Differences by hour

In [ ]:
hourly_summary = (
    df.groupby(["hour", "member_casual"])
      .size()
      .reset_index(name="rides")
)

In [ ]:
plt.figure(figsize=(11, 5))

sns.lineplot(
    data=hourly_summary,
    x="hour",
    y="rides",
    hue="member_casual",
    marker="o"
)

plt.xticks(range(24))
plt.xlabel("Hour of the day")
plt.ylabel("Number of rides")
plt.title("When do members and casual riders use Cyclistic?")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print(
    hourly_summary.loc[
        hourly_summary.groupby("member_casual")["rides"].idxmax()
    ].sort_values("member_casual")
)

###Weekday vs weekend behaviour

In [ ]:
# Create the day_type column (Weekday vs Weekend)
df["day_type"] = df["day_of_week"].apply(
    lambda x: "Weekend" if x in ["Saturday", "Sunday"] else "Weekday"
)

In [ ]:
week_summary = (
    df.groupby(["day_type", "member_casual"])
      .agg(
          rides=("ride_id", "count"),
          average_minutes=("ride_length", "mean")
      )
      .reset_index()
)

In [ ]:
# Calculate the percentage of each rider group's rides
week_summary["within_group_pct"] = (
    week_summary["rides"]
    / week_summary.groupby("member_casual")["rides"].transform("sum")
    * 100
)

print(week_summary.round(2))

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=week_summary,
    x="day_type",
    y="within_group_pct",
    hue="member_casual"
)

plt.title("How each rider group splits rides across the week")
plt.xlabel("")
plt.ylabel("Share of that group's rides (%)")
plt.tight_layout()
plt.show()

###Monthly and seasonal trends

In [ ]:
df["year_month"] = df["started_at"].dt.to_period("M").astype(str)

monthly_summary = (
    df.groupby(["year_month", "member_casual"])
      .size()
      .reset_index(name="rides")
)

In [ ]:
plt.figure(figsize=(13, 5))

sns.lineplot(
    data=monthly_summary,
    x="year_month",
    y="rides",
    hue="member_casual",
    marker="o"
)

plt.title("Monthly Cyclistic ride activity")
plt.xlabel("Month")
plt.ylabel("Number of rides")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
seasons = {12: "Winter", 1: "Winter", 2: "Winter",
           3: "Spring", 4: "Spring", 5: "Spring",
           6: "Summer", 7: "Summer", 8: "Summer",
           9: "Autumn", 10: "Autumn", 11: "Autumn"}
df["season"] = df["started_at"].dt.month.map(seasons)

In [ ]:
season_summary = df.pivot_table(index="season", columns="member_casual", values="ride_id", aggfunc="count")
season_summary = season_summary.reindex(["Spring", "Summer", "Autumn", "Winter"])
season_summary["casual_share_pct"] = (
    season_summary["casual"] / (season_summary["casual"] + season_summary["member"]) * 100
).round(1)
print(season_summary)

In [ ]:
# how much bigger is the busiest month than the quietest one?
monthly_wide = monthly_summary.pivot(index="year_month", columns="member_casual", values="rides")
print("\nBusiest month / quietest month:")
print((monthly_wide.max() / monthly_wide.min()).round(1))

###Ride-duration distribution

In [ ]:
# Ride duration comparison

duration_plot = df[
    (df["ride_length"] > 0) &
    (df["ride_length"] <= 1440) &
    (df["member_casual"].isin(["casual", "member"]))
].copy()

In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=duration_plot,
    x="member_casual",
    y="ride_length",
    order=["casual", "member"],
    showfliers=False
)

plt.title("Ride Duration: Casual Riders vs Members")
plt.xlabel("Rider Type")
plt.ylabel("Ride Duration (Minutes)")
plt.tight_layout()
plt.show()

###Bike-type preferences

In [ ]:
bike_summary = (
    df.groupby(["rideable_type", "member_casual"])
      .size()
      .unstack(fill_value=0)
)

In [ ]:
# Compare preferences within each rider group
bike_share = bike_summary.div(bike_summary.sum(axis=0), axis=1) * 100

print("Ride counts:")
print(bike_summary)

print("\nPercentage of rides by bike type:")
print(bike_share.round(1))

In [ ]:
bike_share.T.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Bike type preference within each rider group")
plt.xlabel("Rider type")
plt.ylabel("Share of rides (%)")
plt.xticks(rotation=0)
plt.legend(title="Bike type")
plt.tight_layout()
plt.show()

###Starting-station patterns

In [ ]:
casual_stations = (
    df.loc[
        (df["member_casual"] == "casual")
        & df["start_station_name"].notna(),
        "start_station_name"
    ]
    .value_counts()
    .head(10)
    .sort_values()
)

In [ ]:
casual_stations.plot(kind="barh", figsize=(9, 6))

plt.title("Top 10 starting stations for casual riders")
plt.xlabel("Number of rides")
plt.ylabel("Starting station")
plt.tight_layout()
plt.show()


In [ ]:
# The same top-10 list for members, so the two can be compared side by side
member_stations = (
    df.loc[(df["member_casual"] == "member") & df["start_station_name"].notna(), "start_station_name"]
    .value_counts()
    .head(10)
    .sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
casual_stations.plot(kind="barh", ax=axes[0], color=colors["casual"])
axes[0].set_title("Top 10 start stations - casual riders")
member_stations.plot(kind="barh", ax=axes[1], color=colors["member"])
axes[1].set_title("Top 10 start stations - members")
for ax in axes:
    ax.set_xlabel("Number of rides")
    ax.set_ylabel("")
plt.tight_layout()
save_chart("top_stations_casual_vs_member")
plt.show()

###Stations with a high casual-rider share

In [ ]:
# Counting rides alone favours busy stations. What really matters for marketing is
# which busy stations are dominated by casual riders - that's where ads will reach them.
station_mix = (
    df.dropna(subset=["start_station_name"])
      .groupby("start_station_name")
      .agg(total_rides=("ride_id", "count"),
           casual_rides=("member_casual", lambda x: (x == "casual").sum()))
)
station_mix["casual_pct"] = station_mix["casual_rides"] / station_mix["total_rides"] * 100

busy_stations = station_mix[station_mix["total_rides"] >= 1000]
print("Busy stations where most riders are casual:", (busy_stations["casual_pct"] > 50).sum())
print(busy_stations.sort_values("casual_rides", ascending=False).head(15).round(1))

###Day and hour heatmap

In [ ]:
# Heatmap: day of week x hour, as % of each group's rides.
# This shows the commute pattern much better than the hour chart alone.
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, rider in zip(axes, ["member", "casual"]):
    table = (
        df[df["member_casual"] == rider]
        .pivot_table(index="day_of_week", columns="hour", values="ride_id", aggfunc="count")
        .reindex(day_order)
        .fillna(0)
    )
    table = table / table.values.sum() * 100
    sns.heatmap(table, cmap="Blues" if rider == "member" else "Oranges", ax=ax,
                cbar_kws={"label": "% of rides"})
    ax.set_title(rider.capitalize())
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("")

plt.suptitle("Members ride at commute hours, casual riders on weekend afternoons", fontweight="bold")
plt.tight_layout()
save_chart("day_hour_heatmap")
plt.show()

Same-station rides and approximate straight-line distance

In [ ]:
# Two behaviour columns: did the bike come back to the same station,
# and how far is it in a straight line from start to end (haversine formula)
df["round_trip"] = df["start_station_name"].notna() & (df["start_station_name"] == df["end_station_name"])

lat1, lon1 = np.radians(df["start_lat"]), np.radians(df["start_lng"])
lat2, lon2 = np.radians(df["end_lat"]), np.radians(df["end_lng"])
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))

behaviour = df.groupby("member_casual").agg(
    round_trip_pct=("round_trip", "mean"),
    median_km=("distance_km", "median"),
    median_minutes=("ride_length", "median"),
)
behaviour["round_trip_pct"] = behaviour["round_trip_pct"] * 100
print(behaviour.round(2))

In [ ]:
# Numbers for the summary table in the Analysis Log
summary = pd.DataFrame({
    "Share of all rides (%)": rider_summary["ride_share_pct"].round(1),
    "Number of rides": rider_summary["total_rides"],
    "Average ride (min)": rider_summary["average_minutes"].round(1),
    "Median ride (min)": rider_summary["median_minutes"].round(1),
    "Busiest day": guide_stats["busiest_day"],
    "Busiest hour": guide_stats["busiest_hour"],
    "Weekend rides (%)": week_summary[week_summary["day_type"] == "Weekend"]
                         .set_index("member_casual")["within_group_pct"].round(1),
    "Summer rides (%)": (df[df["season"] == "Summer"].groupby("member_casual").size()
                         / df.groupby("member_casual").size() * 100).round(1),
    "E-bike rides (%)": bike_share.loc["electric_bike"].round(1),
    "Top start station": [member_stations.index[-1] if g == "member" else casual_stations.index[-1]
                          for g in rider_summary.index],
})
print(summary.T)

## Exploratory Classification model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    balanced_accuracy_score
)

In [ ]:
# Keep plausible durations for this experiment.
# This filter applies to the model sample, not the main analysis.
model_data = df.loc[
    df["ride_length"].between(1, 1440),
    [
        "member_casual",
        "ride_length",
        "hour",
        "day_of_week",
        "month",
        "day_type",
        "rideable_type"
    ]
].copy()

In [ ]:
# Sample enough rides to make the model useful without
# having to train on the entire dataset immediately.
sample_size = min(200_000, len(model_data))

model_data = model_data.sample(
    n=sample_size,
    random_state=42
)

In [ ]:
# Convert duration to a numeric feature in minutes
model_data["ride_length"] = model_data["ride_length"].astype(float)

In [ ]:
X = model_data.drop(columns="member_casual")
y = model_data["member_casual"]

In [ ]:
# Turn the categorical columns into indicator variables
X = pd.get_dummies(
    X,
    columns=["day_of_week", "month", "day_type", "rideable_type"],
    dtype=int
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features used:", X.shape[1])

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=16,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
predictions = rf_model.predict(X_test)

In [ ]:
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 3)
)

print("\nClassification report:")
print(classification_report(y_test, predictions))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    labels=["casual", "member"],
    cmap="Blues",
    values_format=","
)

plt.title("Random Forest: Rider category predictions")
plt.tight_layout()
plt.show()


In [ ]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

top_features = importance.head(10).sort_values()

In [ ]:
plt.figure(figsize=(9, 6))
top_features.plot(kind="barh")

plt.title("Features used by the Random Forest")
plt.xlabel("Feature importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

print(importance.head(10).round(4))


In [ ]:
# MODEL v2 - same Random Forest idea, but stronger and more useful
#
# What changes compared to the first model:
#  1. it also knows WHERE the ride starts and HOW it was ridden (round trip, distance)
#  2. day and month are plain numbers - trees don't need one-hot columns
#  3. it is compared against a dummy baseline, so we know the score means something
#  4. importance is measured by shuffling each feature (permutation importance),
#     which is fairer than the built-in importance that favours numeric columns
#  5. the model is then used for marketing: score every casual ride
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

In [ ]:
def build_features(data):
    features = pd.DataFrame(index=data.index)
    features["ride_length"] = data["ride_length"]
    features["hour"] = data["hour"]
    features["day_num"] = data["started_at"].dt.dayofweek
    features["month_num"] = data["started_at"].dt.month
    features["ebike"] = (data["rideable_type"] == "electric_bike").astype(int)
    features["round_trip"] = data["round_trip"].astype(int)
    features["distance_km"] = data["distance_km"]
    features["start_lat"] = data["start_lat"]
    features["start_lng"] = data["start_lng"]
    return features

In [ ]:
sample2 = df.sample(n=min(300_000, len(df)), random_state=42)
X2 = build_features(sample2)
y2 = sample2["member_casual"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X2_train, y2_train)

rf2 = RandomForestClassifier(
    n_estimators=150,
    max_depth=16,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf2.fit(X2_train, y2_train)

In [ ]:
pred2 = rf2.predict(X2_test)
prob2 = rf2.predict_proba(X2_test)[:, list(rf2.classes_).index("member")]

In [ ]:
print("Balanced accuracy")
print("  baseline (always 'member'):", round(balanced_accuracy_score(y2_test, baseline.predict(X2_test)), 3))
print("  first model               :", round(balanced_accuracy_score(y_test, predictions), 3))
print("  model v2                  :", round(balanced_accuracy_score(y2_test, pred2), 3))
print("ROC-AUC model v2:", round(roc_auc_score(y2_test == "member", prob2), 3))
print()
print(classification_report(y2_test, pred2))

In [ ]:
# Shuffle one column at a time and see how much the model gets worse.
# The bigger the drop, the more that behaviour separates members from casual riders.
check = X2_test.sample(n=min(20_000, len(X2_test)), random_state=42)
perm = permutation_importance(
    rf2, check, y2_test.loc[check.index],
    scoring="balanced_accuracy", n_repeats=5, random_state=42, n_jobs=-1
)
perm_importance = pd.Series(perm.importances_mean, index=check.columns).sort_values()


In [ ]:
plt.figure(figsize=(9, 5))
perm_importance.plot(kind="barh", color=colors["member"])
plt.title("What separates members from casual riders (permutation importance)")
plt.xlabel("Drop in balanced accuracy when the feature is shuffled")
plt.tight_layout()
save_chart("model_v2_importance")
plt.show()

print(perm_importance.sort_values(ascending=False).round(4))

In [ ]:
# THE MARKETING PART
# Give every casual ride a "member-likeness" score (0-100%).
# A casual ride that the model is sure is a member ride comes from someone who
# already uses Cyclistic like a member, just without paying for a membership.
# Those riders are the easiest people to convert.
casual = df[df["member_casual"] == "casual"]
member_col = list(rf2.classes_).index("member")
casual_score = rf2.predict_proba(build_features(casual))[:, member_col]

In [ ]:
likely = casual[casual_score >= 0.7]
likely_pct = len(likely) / len(casual) * 100
print(f"Casual rides that look like member rides (score >= 70%): {len(likely):,} ({likely_pct:.1f}%)")

In [ ]:
# how are these rides different from the average casual ride?
compare = pd.DataFrame({
    "all casual": [casual["ride_length"].median(), (casual["day_type"] == "Weekday").mean() * 100,
                   casual["round_trip"].mean() * 100],
    "member-like casual": [likely["ride_length"].median(), (likely["day_type"] == "Weekday").mean() * 100,
                           likely["round_trip"].mean() * 100],
}, index=["median minutes", "% on weekdays", "% round trips"])
print(compare.round(1))

In [ ]:
target_stations = likely["start_station_name"].value_counts().head(10)
target_hours = likely["hour"].value_counts().sort_index()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(casual_score * 100, bins=40, color=colors["casual"])
axes[0].axvline(70, color="black", linestyle="--")
axes[0].set_title("Member-likeness score of casual rides")
axes[0].set_xlabel("Score (%)")
axes[0].set_ylabel("Casual rides")

target_stations.sort_values().plot(kind="barh", ax=axes[1], color=colors["casual"])
axes[1].set_title("Where they start")
axes[1].set_xlabel("Rides")
axes[1].set_ylabel("")

axes[2].bar(target_hours.index, target_hours.values, color=colors["casual"])
axes[2].set_title("When they ride")
axes[2].set_xlabel("Hour of day")
axes[2].set_xticks(range(0, 24, 3))

plt.suptitle(f"{likely_pct:.0f}% of casual rides already look like member rides - start the campaign here",
             fontweight="bold")
plt.tight_layout()
save_chart("conversion_targets")
plt.show()

In [ ]:
# Save the tables used in the report
pivot_avg = daily_summary.pivot(index="day_of_week", columns="member_casual", values="average_minutes")
pivot_rides = daily_summary.pivot(index="day_of_week", columns="member_casual", values="rides")

In [ ]:
rider_summary.to_csv(output_path + "rider_summary.csv")
guide_stats.to_csv(output_path + "guide_stats.csv")
pivot_avg.to_csv(output_path + "avg_ride_length_by_day.csv")
pivot_rides.to_csv(output_path + "rides_by_day.csv")
season_summary.to_csv(output_path + "season_summary.csv")
busy_stations.sort_values("casual_rides", ascending=False).head(30).to_csv(output_path + "casual_heavy_stations.csv")
perm_importance.sort_values(ascending=False).to_csv(output_path + "model_importance.csv")
target_stations.to_csv(output_path + "conversion_target_stations.csv")
print("Saved to", output_path)

In [ ]:
# ACT - the numbers behind the recommendations
m = rider_summary.loc["member"]
c = rider_summary.loc["casual"]
weekend_share = week_summary.set_index(["member_casual", "day_type"])["within_group_pct"]
summer_casual = season_summary.loc["Summer", "casual"] / season_summary["casual"].sum() * 100
peak_month = monthly_wide["casual"].idxmax()


In [ ]:
print("KEY FINDINGS")
print(f"1. Members take {m['ride_share_pct']:.1f}% of rides, casual riders {c['ride_share_pct']:.1f}%.")
print(f"2. Casual rides are longer: median {c['median_minutes']:.1f} min vs {m['median_minutes']:.1f} min.")
print(f"3. {weekend_share[('casual', 'Weekend')]:.0f}% of casual rides are at weekends "
      f"(members: {weekend_share[('member', 'Weekend')]:.0f}%).")
print(f"4. {summer_casual:.0f}% of casual rides happen in summer; busiest month: {peak_month}.")
print(f"5. Round trips: casual {behaviour.loc['casual', 'round_trip_pct']:.1f}% "
      f"vs member {behaviour.loc['member', 'round_trip_pct']:.1f}%.")
print(f"6. The model tells members from casual riders with balanced accuracy "
      f"{balanced_accuracy_score(y2_test, pred2):.2f}; strongest signal: {perm_importance.index[-1]}.")
print(f"7. {likely_pct:.0f}% of casual rides already look like member rides; "
      f"top station for them: {target_stations.index[0]}.")